In [12]:
import warnings
warnings.filterwarnings("ignore")
# 智普AI
key = 'cb5a18943da54ec389aa9fbd7e20e754.6vVc3r46yQXq7nCD'
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import os

os.environ["ZHIPUAI_API_KEY"] = key
chat = ChatZhipuAI(
    model="glm-4",
    temperature=0.5, #模型温度，值在0-1之间。值越小，随机性越低
)

In [4]:
from langchain_community.embeddings import ZhipuAIEmbeddings
#加载模型
embeddings = ZhipuAIEmbeddings(model="embedding-2")
#准备评论
comments=["这个产品太棒了!","一天就坏了，垃圾!"]
#转换成向量
vectors = embeddings.embed_documents(comments)
print(f"向量的维度:{len(vectors[0])}")
print(f"向量:{vectors[0]}")

向量的维度:1024
向量:[-0.022788005, 0.04684792, 0.0032196764, 0.0765805, 0.03429585, -0.019453475, 0.0026663998, -0.032488313, -0.017323438, 0.025400238, -0.02374617, 0.019384807, -0.017729618, -0.010807459, -0.011479722, 0.04265885, -0.0062909755, -0.026871286, 0.03976138, 0.005519748, -0.01954864, -0.004986424, -0.030890092, -0.038086206, -0.026657892, 0.019591982, -0.0128651075, -0.01623788, 0.031328995, -0.022115009, 0.003531447, -0.01436511, 0.01800431, -0.025245383, 0.022773538, 0.030521655, 0.026309276, 0.029792735, -0.003995673, -0.038266573, 0.007366557, -0.014087699, 0.0032810555, -0.037220966, -0.0039329226, -0.023431178, 0.026054772, 0.09550062, -0.0065815835, 0.03719475, 0.0010141246, 0.019954614, -0.022077477, -0.008320593, 0.038014513, -0.0067519355, 0.057174884, -0.0039632907, -0.022377016, 0.04118003, -0.011748213, 0.05199837, -0.027734937, 0.04648535, -0.012797073, -0.00048448128, 0.017102182, -0.023553135, -0.004075834, -0.022788329, -0.023847375, -0.04410985, -0.025144424,

In [6]:
test_comment=['这个东西超好用!']
test_vector = embeddings.embed_documents(test_comment)
print(f"向量:{test_vector[0]}")

from sklearn.linear_model import LogisticRegression
labels = [1,0] #1好评 0差评
clf = LogisticRegression().fit(vectors,labels)
#预测
prediction = clf.predict(test_vector)
print("好评" if prediction[0]==1 else "差评")

向量:[-0.03441998, 0.025286112, -0.037576757, 0.019262956, 0.012697452, -0.019524377, -0.0048756376, -0.024972314, -0.008113906, 0.033567045, -0.030039512, 0.038675956, -0.031202292, 0.020071875, -0.011749606, 0.032860793, 0.0042226985, -0.04396713, -0.014916448, 0.041733105, -0.007137449, 0.019847414, -0.022989523, -0.029431913, 0.0070243045, -0.013209157, -0.016661707, -0.016209973, 0.03233771, -0.036092356, 0.012022819, -0.004682894, 0.02255585, -0.0075189127, -0.023373153, -0.0048100483, 0.0148087125, 0.037878703, 0.013792458, -0.02317086, -0.00059400365, 0.0038262496, -0.030395297, -0.01160524, -0.0050409045, -0.03860499, 0.013919736, 0.07942702, 0.0152924545, 0.04918866, 0.018115228, 0.024134668, 0.010692832, 0.0351134, 0.025965428, 0.0121038025, 0.020207016, -0.012798403, -0.035928525, 0.04803893, -0.028269337, 0.011054445, -0.043658864, 0.036694508, -0.009311814, 0.004141103, 0.038390238, -0.030192109, -0.004340779, -0.01194176, -0.007851396, 0.0021363501, -0.04808832, -0.0197824

# 本地模型

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

#加载本地模型
embeddings = HuggingFaceEmbeddings(
    model_name='bge-small-zh-v1.5',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': False} # 提升相似度计算精度
)
#测试向量化
query="如何使用langchain"
vectors = embeddings.embed_query(query)
print(f'向量的维度:{len(vectors)}')
print(f'向量的内容:{vectors}')

# 练习

In [9]:
#导入模块
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#加载文件
loader =TextLoader("./政策文件.txt",encoding='utf-8')
documents = loader.load()

#中文分块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,#适合中文段落，段落大小
    chunk_overlap=0,#避免上下缺失
    separators=["\n\n","\n","。","！","？"]#中文常用分隔符
)
splits = text_splitter.split_documents(documents)
print(f'生成{len(splits)}个分块')
print(f'第1块内容:{splits[0]}')
print(f'第1块内容:{splits[0].page_content}')

生成1个分块
第1块内容:page_content='公司年假政策:
1.员工工作满1年可享受5天带薪年假
2.年假有效期至次年3月31日
3.部门经理审批后方可使用
考勤制度:
工作日上班时间:9:00-18:00
迟到超过30分钟记为缺勤' metadata={'source': './政策文件.txt'}
第1块内容:公司年假政策:
1.员工工作满1年可享受5天带薪年假
2.年假有效期至次年3月31日
3.部门经理审批后方可使用
考勤制度:
工作日上班时间:9:00-18:00
迟到超过30分钟记为缺勤


In [13]:
from langchain_community.embeddings import ZhipuAIEmbeddings
#加载模型
embeddings = ZhipuAIEmbeddings(model="embedding-2")
#准备评论
comments=["这个产品太棒了!","一天就坏了，垃圾!"]
#转换成向量
vectors = embeddings.embed_documents(comments)
print(f"向量的维度:{len(vectors[0])}")
print(f"向量:{vectors[0]}")

向量的维度:1024
向量:[-0.022788005, 0.04684792, 0.0032196764, 0.0765805, 0.03429585, -0.019453475, 0.0026663998, -0.032488313, -0.017323438, 0.025400238, -0.02374617, 0.019384807, -0.017729618, -0.010807459, -0.011479722, 0.04265885, -0.0062909755, -0.026871286, 0.03976138, 0.005519748, -0.01954864, -0.004986424, -0.030890092, -0.038086206, -0.026657892, 0.019591982, -0.0128651075, -0.01623788, 0.031328995, -0.022115009, 0.003531447, -0.01436511, 0.01800431, -0.025245383, 0.022773538, 0.030521655, 0.026309276, 0.029792735, -0.003995673, -0.038266573, 0.007366557, -0.014087699, 0.0032810555, -0.037220966, -0.0039329226, -0.023431178, 0.026054772, 0.09550062, -0.0065815835, 0.03719475, 0.0010141246, 0.019954614, -0.022077477, -0.008320593, 0.038014513, -0.0067519355, 0.057174884, -0.0039632907, -0.022377016, 0.04118003, -0.011748213, 0.05199837, -0.027734937, 0.04648535, -0.012797073, -0.00048448128, 0.017102182, -0.023553135, -0.004075834, -0.022788329, -0.023847375, -0.04410985, -0.025144424,

In [17]:
from langchain_community.vectorstores import Chroma
db = Chroma.from_documents(
    documents=splits,
    embedding=embeddings, # 使用本地模型生成向量 / 也可改成网络模型测试
    persist_directory="./chromadb"
)
#语义搜索
docs = db.similarity_search("迟到处罚",k=2)
print('匹配结果:')
for doc in docs:
    print(doc.page_content)

匹配结果:
公司年假政策:
1.员工工作满1年可享受5天带薪年假
2.年假有效期至次年3月31日
3.部门经理审批后方可使用
考勤制度:
工作日上班时间:9:00-18:00
迟到超过30分钟记为缺勤
